In [1]:
import os
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from sklearn.model_selection import train_test_split

# Define processing variables

In [2]:
# These parameters have been separated since they are important for the experimentation the buffer defines the sequence length of the input to the neural network
# Different buffers have been used with different samples rates these are:
# 256 buffer length with 1 sample rate | sequence length of 256 samples with no downsampling
# 128 buffer length with 1 sample rate | sequence length of 128 samples with no downsampling
# 64 buffer length with 1 sample rate  | sequence length of 64 samples with no downsampling
# 32 buffer length with 1 sample rate  | sequence length of 32 samples with no downsampling
# 64 buffer length with 2 sample rate  | sequence length of 32 samples with downsampling by a factor of 2
# 128 buffer length with 2 sample rate | sequence length of 64 samples with downsampling by a factor of 2
# 256 buffer length with 2 sample rate | sequence length of 128 samples with downsampling by a factor of 2
# 512 buffer length with 2 sample rate | sequence length of 256 samples with downsampling by a factor of 2
# 1024 buffer length with 4 sample rate | sequence length of 256 samples with downsampling by a factor of 4
# 1024 buffer length with 1 sample rate | sequence length of 1024 samples with no downsampling (for LSTM model)

buf = 256                                   # Size of input to CNN in number of I/Q samples, this is necessary since the wave contains multiple frequencies
samples_rate = 4                            # To downsample the samples (if no downsampling desired set samples_rate = 1)

In [3]:
# Set good samples
GOOD_SAMPLES = [
    '0001_sample1.dat',
    '1000_sample1.dat',
    '1000_sample8.dat',
    '0001_sample2.dat',
    '0001_sample3.dat',
    '0001_sample4.dat',
    '0001_sample5.dat',
    '0010_sample1.dat',
    '0010_sample3.dat',
    '0010_sample4.dat',
    '0100_sample1.dat',
    '0100_sample2.dat',
    '1000_sample2.dat',
    '1000_sample3.dat',
    '1000_sample4.dat',
    '1000_sample5.dat',
    '1000_sample6.dat',
    # Higher noise samples
    '1100_sample1.dat',
    '1100_sample2.dat',
    '1010_sample1.dat',
]

# If you desire to plot the spectrogram of the samples set below to True
plot_spect = False
plot_log = False
use_good_samples = False
use_new_directory = False

# Getting list of raw .bin files
if use_new_directory:
    experiment_bin_folder_fp = "../data/experiment/binary/new/"
else:
    experiment_bin_folder_fp = "../data/experiment/binary/"

if use_good_samples:
    experiment_bin_folder = [f for f in os.listdir(experiment_bin_folder_fp) if f in GOOD_SAMPLES]  # list of files in folder
else:
    experiment_bin_folder = os.listdir(experiment_bin_folder_fp)      # list of files in folder


print(f"{len(experiment_bin_folder)} files found in {experiment_bin_folder_fp}")

# Filepath of folder that will contain the converted h5 files
h5_folder_fp = "../data/experiment/h5/"

if not os.path.isdir(h5_folder_fp):
    os.mkdir(h5_folder_fp)

# Parameters for training models

# To create overlap between samples (if no overlap desired set stride = buf)"
stride = 16                           

# Number of buf sized training/testing samples to be gathered from each .bin file
nsamples_per_file = 15_000          

# Number of complex values to read from .bin file to generate desired amount of training/testing samples
# If you want to read all the complex values set niq2read = -1
niq2read = (nsamples_per_file - 1) * stride + buf
# niq2read = -1

# Number of complex values to skip over before reading
offset = 0


84 files found in ../data/experiment/binary/


# Convert from binary to h5

In [4]:
def plot_spectrogram(filename, samps, plot_log=True):
    # Generate spectrogram at sampling rate of 20MHz
    f, t, Sxx = spectrogram(
        x=samps, 
        fs=20000000, 
        return_onesided=False
    )

    # Compensate for FFT Shift caused by GNU Radio
    Sxx = np.fft.fftshift(x=Sxx, axes=0)
    f = np.fft.fftshift(f)

    if plot_log:
        Sxx = 10 * np.log10(Sxx)  # Convert to dB scale

    # Plot spectrogram with dual x-axis
    fig, ax = plt.subplots()

    if plot_log:
        pcm = ax.pcolormesh(t, f, Sxx, shading='auto', vmin=-140, vmax=-80, cmap='rainbow')
    else:
        pcm = ax.pcolormesh(t, f, Sxx, shading='auto', vmin=0, vmax=0.5e-13, cmap='rainbow')


    ax.set_ylabel('Frequency [Hz]')
    ax.set_xlabel('Time [sec]')
    fig.colorbar(pcm, ax=ax, label='Power Spectral Density [dB]')
    ax.set_title(filename + ' log' if plot_log else filename)

    # Add top x-axis with sequence number
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    step = max(1, len(t) // 10)
    ax_top.set_xticks(t[::step])
    ax_top.set_xticklabels(range(len(t))[::step])
    ax_top.set_xlabel('Sample Number')

    ax.set_yticks([
        0.75 * 1e7, 
        0.625 * 1e7,
        0.5 * 1e7,
        0.375 * 1e7,
        0.25 * 1e7, 
        0.125 * 1e7,
        0,
        -0.125 * 1e7,
        -0.25 * 1e7, 
        -0.375 * 1e7,
        -0.5 * 1e7,
        -0.625 * 1e7,
        -0.75 * 1e7
    ])

    # Change y-axis left to channel numbers
    ax.set_yticklabels([
        'Channel 1',
        'Channel 2',
        'Channel 3',
        'Channel 4',
        'Channel 5',
        'Channel 6',
        'Channel 7',
        'Channel 8',
        'Channel 9',
        'Channel 10',
        'Channel 11',
        'Channel 12',
        'Channel 13',
        # 'Channel 14',
    ])

    plt.tight_layout()
    plt.show()
    

    # Add top x-axis with sequence number
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    step = max(1, len(t) // 10)
    ax_top.set_xticks(t[::step])
    ax_top.set_xticklabels(range(len(t))[::step])
    ax_top.set_xlabel('Sequence Number')
    plt.tight_layout()
    plt.show()

In [5]:
# Iterate through each .bin file and add contents to .h5 file, which is an efficient data storage data format
for filename in experiment_bin_folder:
    filepath = os.path.join(experiment_bin_folder_fp, filename)

    # Check if the file is a directory then skip
    if os.path.isdir(filepath):
        continue

    with open(filepath) as binary_file:
        # Extract desired number of samples
        samps = np.fromfile(binary_file, dtype=np.complex64, count=niq2read, offset=offset)

        # Plot samples
        if plot_spect:
            if plot_log:
                plot_spectrogram(filename, samps, plot_log=True)
            plot_spectrogram(filename, samps, plot_log=False)

        # Turn 1D complex array of raw samples and reshape into a 2D array containing I and Q as floats
        # Get real and imaginary parts of samples and stack them together
        samps = np.transpose(np.stack((np.real(samps), np.imag(samps))))

        # Break long 2D array containing all I/Q values into multiple training/testing samples
        samps = np.array([samps[k:k + buf:samples_rate] for k in range(0, len(samps) - 1 - buf, stride)])

        # Create .h5 file with same name as .bin file and fill with reshaped samples
        name = os.path.splitext(filename)[0]
        f = h5py.File(h5_folder_fp + name + '.h5', 'w')
        # Create dataset with same name as .bin file
        dset = f.create_dataset(name, (samps.shape[0], samps.shape[1], samps.shape[2]), dtype='float32')
        # Fill dataset with reshaped samples
        dset[()] = samps

        # Print shape
        print(f"File: {name}.h5, Shape: {samps.shape}")
        f.close()

# source
# elka

File: 0000_sample1.h5, Shape: (14999, 64, 2)
File: 0000_sample2.h5, Shape: (14999, 64, 2)
File: 0000_sample3.h5, Shape: (14999, 64, 2)
File: 0000_sample4.h5, Shape: (14999, 64, 2)
File: 0000_sample5.h5, Shape: (14999, 64, 2)
File: 0000_sample6.h5, Shape: (14999, 64, 2)
File: 0000_sample7.h5, Shape: (14999, 64, 2)
File: 0000_sample8.h5, Shape: (14999, 64, 2)
File: 0000_sample9.h5, Shape: (14999, 64, 2)
File: 0001_sample1.h5, Shape: (14999, 64, 2)
File: 0001_sample3.h5, Shape: (14999, 64, 2)
File: 0001_sample4.h5, Shape: (14999, 64, 2)
File: 0010_sample1.h5, Shape: (14999, 64, 2)
File: 0010_sample10.h5, Shape: (14999, 64, 2)
File: 0010_sample11.h5, Shape: (14999, 64, 2)
File: 0010_sample12.h5, Shape: (14999, 64, 2)
File: 0010_sample2.h5, Shape: (14999, 64, 2)
File: 0010_sample4.h5, Shape: (14999, 64, 2)
File: 0010_sample5.h5, Shape: (14999, 64, 2)
File: 0010_sample6.h5, Shape: (14999, 64, 2)
File: 0010_sample7.h5, Shape: (14999, 64, 2)
File: 0010_sample8.h5, Shape: (14999, 64, 2)
File: 0

In [6]:
# Filepath containing directory with converted .h5 files
h5_folder_fp = "../data/experiment/h5/"
folder = os.listdir(h5_folder_fp)
folder.sort()

# Generate dummy arrays to be contain entire dataset and dataset labels (can also use list and convert to np.array later) (labels are for for channels if occupied or not)
dataset_labels = np.zeros((1, 4))
dataset = np.zeros((1, buf // samples_rate, 2))

for filepath in folder: 
    if not os.path.isdir(h5_folder_fp + filepath):
        # Open .h5 file
        f = h5py.File(h5_folder_fp + filepath, 'r')
        # Get label from filename
        name = os.path.splitext(filepath)[0]
        # Get data from file
        data = f[name][()]

        # Append samples from current file to dataset
        dataset = np.concatenate((dataset, data))

        # Generates the multi-hot encoded labels from the file name
        label = list(name.split('_')[0])    # Take part of filename that contains labels
        label = list(map(int, label))       # Convert string to multi-hot list
        label = [label] * data.shape[0]     # Generate label for each training sample in file
        label = np.array(label, dtype='i')  # Convert list of labels to np.array
        dataset_labels = np.concatenate((dataset_labels, label))    # Append to labels for entire dataset

        print(f"File: {name}.h5, Shape: {data.shape}")

f.close()

# Delete first entry of arrays as they contain zeros
dataset = np.delete(dataset, 0, 0)
dataset_labels = np.delete(dataset_labels, 0, 0)

print(dataset.shape)
print(dataset_labels.shape)

File: 0000_sample1.h5, Shape: (14999, 64, 2)
File: 0000_sample2.h5, Shape: (14999, 64, 2)
File: 0000_sample3.h5, Shape: (14999, 64, 2)
File: 0000_sample4.h5, Shape: (14999, 64, 2)
File: 0000_sample5.h5, Shape: (14999, 64, 2)
File: 0000_sample6.h5, Shape: (14999, 64, 2)
File: 0000_sample7.h5, Shape: (14999, 64, 2)
File: 0000_sample8.h5, Shape: (14999, 64, 2)
File: 0000_sample9.h5, Shape: (14999, 64, 2)
File: 0001_sample1.h5, Shape: (14999, 64, 2)
File: 0001_sample3.h5, Shape: (14999, 64, 2)
File: 0001_sample4.h5, Shape: (14999, 64, 2)
File: 0010_sample1.h5, Shape: (14999, 64, 2)
File: 0010_sample10.h5, Shape: (14999, 64, 2)
File: 0010_sample11.h5, Shape: (14999, 64, 2)
File: 0010_sample12.h5, Shape: (14999, 64, 2)
File: 0010_sample2.h5, Shape: (14999, 64, 2)
File: 0010_sample4.h5, Shape: (14999, 64, 2)
File: 0010_sample5.h5, Shape: (14999, 64, 2)
File: 0010_sample6.h5, Shape: (14999, 64, 2)
File: 0010_sample7.h5, Shape: (14999, 64, 2)
File: 0010_sample8.h5, Shape: (14999, 64, 2)
File: 0

In [7]:
# Shuffle dataset and split into training and testing samples
X_train, X_test, y_train, y_test = train_test_split(
    dataset, 
    dataset_labels, 
    test_size=0.1, 
    random_state=42
)

# Print shapes of training and testing samples
print("X_train shape: ", X_train.shape)
print("y_train shape: ", y_train.shape)
print("X_test shape: ", X_test.shape)
print("y_test shape: ", y_test.shape)

X_train shape:  (1120425, 64, 2)
y_train shape:  (1120425, 4)
X_test shape:  (124492, 64, 2)
y_test shape:  (124492, 4)


# Data exploration

In [8]:
print('Sample of training set:')
print(X_train[0])
print('Label of training set:')
print(y_train[0])

Sample of training set:
[[ 6.10351562e-04 -3.05175781e-04]
 [ 7.93457031e-04 -1.83105469e-04]
 [ 4.57763672e-04  1.12915039e-03]
 [ 1.22070312e-04  1.22070312e-04]
 [-1.86157227e-03  7.32421875e-04]
 [ 4.88281250e-04 -2.13623047e-04]
 [-5.79833984e-04 -5.79833984e-04]
 [ 1.83105469e-04 -1.31225586e-03]
 [ 6.40869141e-04 -1.83105469e-04]
 [ 7.32421875e-04 -7.62939453e-04]
 [-3.35693359e-04  6.71386719e-04]
 [ 5.18798828e-04  6.10351562e-04]
 [-1.15966797e-03  2.13623047e-04]
 [-9.15527344e-04  4.27246094e-04]
 [ 9.15527344e-05 -7.93457031e-04]
 [-2.13623047e-04 -4.27246094e-04]
 [ 0.00000000e+00  9.15527344e-05]
 [ 5.49316406e-04  8.23974609e-04]
 [ 5.79833984e-04 -2.13623047e-04]
 [ 1.52587891e-04  3.66210938e-04]
 [-1.40380859e-03  5.18798828e-04]
 [-2.44140625e-04  1.19018555e-03]
 [-1.00708008e-03 -1.43432617e-03]
 [-6.10351562e-05 -1.00708008e-03]
 [-6.40869141e-04 -3.96728516e-04]
 [ 1.12915039e-03 -4.27246094e-04]
 [ 7.32421875e-04  9.46044922e-04]
 [-9.15527344e-05  4.27246094e-

In [9]:
print()

In [10]:
# Filter for label
label = [0,1,1,0]

# Find index of first sample with label
idx = np.where((y_train == label).all(axis=1))[0]

for sam_i in idx[:16]:
    # Define parameters
    real_signal = X_train[sam_i, :, 0] # Shape: (128,1) so array of 128 real values
    imag_signal = X_train[sam_i, :, 1] # Shape: (128,1) so array of 128 imaginary values

    # Combine real and imaginary parts to form a complex signal 1D array
    complex_signal = real_signal + 1j * imag_signal

    # Generate spectrogram at sampling rate of 20MHz
    f, t, Sxx = spectrogram(
        x=complex_signal,
        fs=20000000,
        nperseg=32,
        noverlap=24,
        return_onesided=False
    )

    # Compensate for FFT Shift caused by GNU Radio
    Sxx = np.fft.fftshift(
        x=Sxx, 
        axes=0
    )
    f = np.fft.fftshift(f)

    if plot_spect:
        fig, ax = plt.subplots()

        # Plot spectrogram
        pcm = ax.pcolormesh(t, f, Sxx, shading='auto', vmax=6 * 1e-11, cmap='coolwarm')
        ax.set_ylabel('Frequency [Hz]')
        ax.set_xlabel('Time [sec]')
        fig.colorbar(pcm, ax=ax, label='Power Spectral Density [dB]')
        ax.set_title(str(y_train[sam_i]))

        # Add top x-axis for sequence numbers
        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())
        step = 1 # max(1, len(t) // 4)
        ax_top.set_xticks(t[::step])
        ax_top.set_xticklabels(range(len(t))[::step])
        ax_top.set_xlabel('Sequence Number')

    # Get the signal strength over 4 channels
    channel_size = len(f) // 4
    channels = np.zeros(4)

    for i in range(4):
        channel_num = i + 1

        # Combine all the frequencies in the channel
        channel_strength = Sxx[i*channel_size:(i+1)*channel_size]
        # Sum all values in the channel to get occupancy of channel
        channel_strength = np.sum(channel_strength)
        # Make values from 0 to 1 on relative scale to each other
        channel_strength = channel_strength / np.sum(Sxx)

        channels[i] = channel_strength

    print(f'Channels: {channels}')
    print(f'Label: {y_train[sam_i]}')
    print(f'Shape: {Sxx.shape}')

Channels: [0.01387181 0.51480921 0.36279878 0.1085202 ]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.34126551 0.20876381 0.14450652 0.30546417]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.25417944 0.27568019 0.17385254 0.29628783]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.39206614 0.3161601  0.07519209 0.21658167]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.20627838 0.06061383 0.68329294 0.04981485]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.35285153 0.27584785 0.11892474 0.25237588]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.39633201 0.14703151 0.15865092 0.29798556]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.28615344 0.23311333 0.23231668 0.24841655]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.34056277 0.08728628 0.05847709 0.51367386]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.28701171 0.13881573 0.11361249 0.46056007]
Label: [0. 1. 1. 0.]
Shape: (32, 5)
Channels: [0.30494504 0.21804548 0.12236322 0.35464625]
Label: [0. 1. 1. 0.]
Sha

# Generating data

In [11]:
def generate_spectrogram_dataset(data):
    result = None

    for sam_i in range(data.shape[0]):
        sys.stdout.write(f'\rProcessing sample {sam_i+1}/{data.shape[0]}')

        # Define parameters
        real_signal = data[sam_i, :, 0] # Shape: (128,1) so array of 128 real values
        imag_signal = data[sam_i, :, 1] # Shape: (128,1) so array of 128 imaginary values

        # Combine real and imaginary parts to form a complex signal 1D array
        complex_signal = real_signal + 1j * imag_signal

        # Generate spectrogram at sampling rate of 20MHz
        f, t, Sxx = spectrogram(
            x=complex_signal,
            fs=20000000,
            nperseg=32,
            noverlap=24,
            return_onesided=False
        )

        # Compensate for FFT Shift caused by GNU Radio
        Sxx = np.fft.fftshift(
            x=Sxx, 
            axes=0
        )
        f = np.fft.fftshift(f)

        if plot_spect:
            # Plot spectrogram
            plt.pcolormesh(t, f, Sxx, shading='auto', vmax=np.max(Sxx)/100, cmap='coolwarm')
            plt.ylabel('Frequency [Hz]')
            plt.xlabel('Time [sec]')
            plt.colorbar(label='Power Spectral Density [dB]')
            plt.title(y_train[sam_i])
            plt.show()

        # Get the signal strength over 4 channels
        channel_size = len(f) // 4
        channels = np.zeros((4, len(t)))

        for i in range(4):
            # Combine all the frequencies in the channel
            channel_strength = Sxx[i*channel_size:(i+1)*channel_size]

            # Change from (15, 8) to (1, 8) by summing over the frequency axis
            channel_strength = np.sum(channel_strength, axis=0)

            # Save the channel strength to correct channel
            channels[i] = channel_strength

        # Average out channels between 0 and 1
        for n in range(channels.shape[1]):
            channels[:, n] = channels[:, n] / np.sum(channels[:, n])

        if result is None:
            result = np.zeros((data.shape[0], 4, len(t)))
        
        result[sam_i] = channels

    return result

In [12]:
CREATE_SPECTROGRAM = False

if CREATE_SPECTROGRAM:
    X_train_spectrogram = generate_spectrogram_dataset(X_train)
    print(X_train_spectrogram.shape)
    X_test_spectrogram = generate_spectrogram_dataset(X_test)
    print(X_test_spectrogram.shape)

In [13]:
# Save test set
f_test = h5py.File('../data/experiment/sdr_wifi_test.hdf5', 'w')
xtest = f_test.create_dataset(
    name='X', 
    shape=X_test.shape,
    dtype='f',
    data=X_test
)
ytest = f_test.create_dataset(
    name='y', 
    shape=y_test.shape,
    dtype='i',
    data=y_test
)
f_test.close()

# Save train set
f_train = h5py.File('../data/experiment/sdr_wifi_train.hdf5', 'w')
xtrain = f_train.create_dataset(
    name='X', 
    shape=X_train.shape,
    dtype='f',
    data=X_train
)
ytrain = f_train.create_dataset(
    name='y', 
    shape=y_train.shape,
    dtype='i',
    data=y_train
)
f_train.close()

if CREATE_SPECTROGRAM:
    # Save test set with spectrogram
    f_test_spectrogram = h5py.File('../data/sdr_wifi_test_spectrogram.hdf5', 'w')
    xtest_spectrogram = f_test_spectrogram.create_dataset(
        name='X', 
        shape=X_test_spectrogram.shape,
        dtype='f',
        data=X_test_spectrogram
    )
    ytest_spectrogram = f_test_spectrogram.create_dataset(
        name='y', 
        shape=y_test.shape,
        dtype='i',
        data=y_test
    )
    f_test_spectrogram.close()

    # Save train set with spectrogram
    f_train_spectrogram = h5py.File('../data/sdr_wifi_train_spectrogram.hdf5', 'w')
    xtrain_spectrogram = f_train_spectrogram.create_dataset(
        name='X', 
        shape=X_train_spectrogram.shape,
        dtype='f',
        data=X_train_spectrogram
    )
    ytrain_spectrogram = f_train_spectrogram.create_dataset(
        name='y', 
        shape=y_train.shape,
        dtype='i',
        data=y_train
    )
    f_train_spectrogram.close()